# 03 — Hypothesis-Driven Feature Engineering
**Project:** Crop Yield Prediction and Forecasting Using Agricultural Data and Machine Learning  
*Author: Aryan Puranik*

This notebook constructs hypothesis-driven feature groups for next-year crop yield forecasting:
$$\text{Predict } \text{Yield}_{r,c,t+1} \text{ using strictly pre-forecast information available up to year } t.$$

### Research Objectives
1. **Exact-Calendar Historical Lags:** Construct causal yield lags ($Y_t, Y_{t-1}, Y_{t-2}, Y_{t-4}$) and rolling statistics over previous harvests.
2. **Outlier-Sanitized History Lookup:** Mask candidate 2007 Chickpea data-entry typos (>10,000 kg/ha) so they never pollute subsequent lag features.
3. **District-Relative Climate Anomalies:** Compute causal climate normals up to year $t-1$ to isolate true local weather shocks from cross-sectional geographic baselines.
4. **Regional Productivity Baselines:** Calculate long-term district-crop median productivity strictly from training records (Target_Year $\le 2011$).
5. **Deduplicated Region-Year Climate Panel:** Generate an independent climate panel to support Phase 06 climate forecasting experiments without duplicating weather records across crops.
6. **Heuristic Baseline Ladder:** Benchmark naive persistence, rolling 3-year mean, and trend-adjusted naives on the 2012–2014 validation cohort.
7. **No Predictive ML Model Fitting:** In strict adherence to Rule 1, no machine learning regression models are trained here.


## 1. Setup, Imports, and Data Loading

We load the prepared next-year forecasting panels generated in Phase 02 (`train_forecast_panel.csv`, `val_forecast_panel.csv`, `test_forecast_panel.csv`) and the raw dataset for historical lag extraction.


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "crop_yield_dataset.csv"

train_panel = pd.read_csv(PROCESSED_DIR / "train_forecast_panel.csv")
val_panel = pd.read_csv(PROCESSED_DIR / "val_forecast_panel.csv")
test_panel = pd.read_csv(PROCESSED_DIR / "test_forecast_panel.csv")
raw_df = pd.read_csv(RAW_PATH)

print(f"Loaded Panels:")
print(f"  Training:   {train_panel.shape[0]:,} rows (Target Years: {train_panel['Target_Year'].min()}-{train_panel['Target_Year'].max()})")
print(f"  Validation: {val_panel.shape[0]:,} rows (Target Years: {val_panel['Target_Year'].min()}-{val_panel['Target_Year'].max()})")
print(f"  Test:       {test_panel.shape[0]:,} rows (Target Years: {test_panel['Target_Year'].min()}-{test_panel['Target_Year'].max()})")


Loaded Panels:
  Training:   164,915 rows (Target Years: 1967-2011)
  Validation: 10,610 rows (Target Years: 2012-2014)
  Test:       10,160 rows (Target Years: 2015-2017)


## 2. Outlier-Sanitized History Lookup Series

To prevent known 2007 Chickpea data-entry typos (e.g., Muzaffarpur 58,363 kg/ha and Saharsa 32,000 kg/ha) from corrupting future historical features ($Y_{t-1}$ in 2008, 3-year rolling means in 2008–2010, and multi-year trends in 2012), we create a sanitized historical lookup source where these impossible values are replaced with `NaN`.


In [4]:
clean_lookup = raw_df.copy()
chickpea_outliers = (
    (clean_lookup["Crop"] == "Chickpea") &
    (clean_lookup["Year"] == 2007) &
    (clean_lookup["Yield"] > 10000)
)
print(f"Masked {chickpea_outliers.sum()} anomalous Chickpea 2007 entries in historical lookup series.")
clean_lookup.loc[chickpea_outliers, "Yield"] = np.nan


Masked 2 anomalous Chickpea 2007 entries in historical lookup series.


## 3. Construct Hypothesis-Driven Feature Groups

We engineer 10 controlled feature groups for each panel:
1. **Historical Yield Lags relative to Target $t+1$:**
   - $\text{Yield\_Lag1} = Y_t$ (most recent observed harvest at forecast origin)
   - $\text{Yield\_Lag2} = Y_{t-1}$ (harvest 2 years prior to target)
   - $\text{Yield\_Lag3} = Y_{t-2}$ (harvest 3 years prior to target)
   - $\text{Yield\_Lag5} = Y_{t-4}$ (harvest 5 years prior to target)
2. **Multi-Year Rolling Statistics:**
   - $\text{Yield\_Rolling3} = \text{mean}(Y_t, Y_{t-1}, Y_{t-2})$
   - $\text{Yield\_Rolling3\_Std} = \text{std}(Y_t, Y_{t-1}, Y_{t-2})$
   - $\text{Yield\_Rolling5} = \text{mean}(Y_t, Y_{t-1}, Y_{t-2}, Y_{t-3}, Y_{t-4})$
3. **Trends and Momentum:**
   - $\text{Yield\_Change\_5Y} = Y_t - Y_{t-4}$
   - $\text{Yield\_Trend\_Velocity} = (Y_t - Y_{t-4}) / 4.0$
   - $\text{Yield\_Trend\_Naive} = Y_t + (Y_t - Y_{t-2}) / 2.0$
4. **Historical Climate Levels & Causal Anomalies:**
   - Origin climate: $\text{Temperature}_t, \text{Rainfall}_t, \text{Humidity}_t$
   - Expanding historical normals strictly up to year $t-1$
   - Anomalies: $\text{Temperature\_Anomaly}, \text{Rainfall\_Anomaly}, \text{Humidity\_Anomaly}$
5. **Agronomic Interactions & Balance:**
   - $\text{NPK\_Total} = N + P + K$
   - $\text{N\_P\_Ratio} = N / (P + 1e-4)$
   - $\text{Rain\_Temp\_Ratio} = \text{Rainfall}_t / (\text{Temperature}_t + 1e-4)$
6. **Regional Productivity Baseline (Training-Only Learned):**
   - District-crop median yield learned strictly from training data ($t+1 \le 2011$)
   - Relative productivity index: $Y_t / (\bar{Y}_{r,c}^{\text{train}} + 1e-4)$
7. **Availability & Temporal Trend Flags:**
   - $\text{History\_Lag1\_Available}, \text{History\_3yr\_Complete}, \text{History\_5yr\_Complete}$
   - $\text{Is\_Post\_1990} = (\text{Year} \ge 1990)$


In [6]:
# 1. Compute regional productivity baselines strictly on training records
reg_crop_median = (
    train_panel.groupby(["Region", "Crop"])["Yield"]
    .median()
    .rename("Hist_Region_Crop_Median")
)
crop_median = (
    train_panel.groupby("Crop")["Yield"]
    .median()
    .rename("Hist_Crop_Median")
)

# 2. Compute causal climate normals using expanding window strictly up to year t-1
climate_df = raw_df[["Region", "Crop", "Year", "Temperature", "Rainfall", "Humidity"]].sort_values(["Region", "Crop", "Year"]).copy()
climate_df["Temp_shift"] = climate_df.groupby(["Region", "Crop"])["Temperature"].shift(1)
climate_df["Rain_shift"] = climate_df.groupby(["Region", "Crop"])["Rainfall"].shift(1)
climate_df["Hum_shift"] = climate_df.groupby(["Region", "Crop"])["Humidity"].shift(1)

climate_df["Temp_Normal"] = climate_df.groupby(["Region", "Crop"])["Temp_shift"].expanding(min_periods=3).mean().values
climate_df["Rain_Normal"] = climate_df.groupby(["Region", "Crop"])["Rain_shift"].expanding(min_periods=3).mean().values
climate_df["Hum_Normal"] = climate_df.groupby(["Region", "Crop"])["Hum_shift"].expanding(min_periods=3).mean().values

climate_df["Temperature_Anomaly"] = climate_df["Temperature"] - climate_df["Temp_Normal"]
climate_df["Rainfall_Anomaly"] = climate_df["Rainfall"] - climate_df["Rain_Normal"]
climate_df["Humidity_Anomaly"] = climate_df["Humidity"] - climate_df["Hum_Normal"]

climate_features = climate_df[[
    "Region", "Crop", "Year",
    "Temperature_Anomaly", "Rainfall_Anomaly", "Humidity_Anomaly"
]]

def enrich_panel(panel_df):
    p = panel_df.copy()
    
    # Lag 1 is the yield observed at forecast origin year t
    p["Yield_Lag1"] = p["Yield"]
    
    # Lags 2, 3, 4, 5
    for lag, offset in [(2, 1), (3, 2), (4, 3), (5, 4)]:
        prev = clean_lookup[["Region", "Crop", "Year", "Yield"]].rename(
            columns={"Year": f"Prev_Year_{lag}", "Yield": f"Yield_Lag{lag}"}
        )
        p[f"Prev_Year_{lag}"] = p["Year"] - offset
        p = p.merge(prev, on=["Region", "Crop", f"Prev_Year_{lag}"], how="left").drop(columns=[f"Prev_Year_{lag}"])
    
    # Rolling 3-year and 5-year statistics
    lags_3 = ["Yield_Lag1", "Yield_Lag2", "Yield_Lag3"]
    lags_5 = ["Yield_Lag1", "Yield_Lag2", "Yield_Lag3", "Yield_Lag4", "Yield_Lag5"]
    p["Yield_Rolling3"] = p[lags_3].mean(axis=1, skipna=False)
    p["Yield_Rolling3_Std"] = p[lags_3].std(axis=1, ddof=0, skipna=False)
    p["Yield_Rolling5"] = p[lags_5].mean(axis=1, skipna=False)
    
    # Trends and velocities
    p["Yield_Change_5Y"] = p["Yield_Lag1"] - p["Yield_Lag5"]
    p["Yield_Trend_Velocity"] = (p["Yield_Lag1"] - p["Yield_Lag5"]) / 4.0
    p["Yield_Trend_Naive"] = p["Yield_Lag1"] + (p["Yield_Lag1"] - p["Yield_Lag3"]) / 2.0
    
    # Merge climate anomalies
    p = p.merge(climate_features, on=["Region", "Crop", "Year"], how="left")
    
    # Merge training-learned regional productivity baselines
    p = p.merge(reg_crop_median, on=["Region", "Crop"], how="left")
    p = p.merge(crop_median, on=["Crop"], how="left")
    p["Hist_Region_Crop_Median"] = p["Hist_Region_Crop_Median"].fillna(p["Hist_Crop_Median"])
    p["Yield_Relative_Index"] = p["Yield_Lag1"] / (p["Hist_Region_Crop_Median"] + 1e-4)
    
    # Agronomic interactions
    p["NPK_Total"] = p["Nitrogen"] + p["Phosphorus"] + p["Potassium"]
    p["N_P_Ratio"] = p["Nitrogen"] / (p["Phosphorus"] + 1e-4)
    p["Rain_Temp_Ratio"] = p["Rainfall"] / (p["Temperature"] + 1e-4)
    
    # Availability indicators and era flag
    p["History_Lag1_Available"] = p["Yield_Lag1"].notna().astype(int)
    p["History_3yr_Complete"] = p[lags_3].notna().all(axis=1).astype(int)
    p["History_5yr_Complete"] = p[lags_5].notna().all(axis=1).astype(int)
    p["Is_Post_1990"] = (p["Year"] >= 1990).astype(int)
    
    return p

train_enriched = enrich_panel(train_panel)
val_enriched = enrich_panel(val_panel)
test_enriched = enrich_panel(test_panel)

print(f"Enriched Panels:")
print(f"  Training:   {train_enriched.shape[0]:,} rows x {train_enriched.shape[1]} columns")
print(f"  Validation: {val_enriched.shape[0]:,} rows x {val_enriched.shape[1]} columns")
print(f"  Test:       {test_enriched.shape[0]:,} rows x {test_enriched.shape[1]} columns")
train_enriched[["Region", "Crop", "Year", "Target_Year", "Target_Yield", "Yield_Lag1", "Yield_Rolling3", "Temperature_Anomaly"]].head(3)


Enriched Panels:
  Training:   164,915 rows x 42 columns
  Validation: 10,610 rows x 42 columns
  Test:       10,160 rows x 42 columns


## 4. Deduplicated Region-Year Climate Panel (For Phase 06)

In raw agricultural data, climate variables (`Temperature`, `Rainfall`, `Humidity`) are identical across all crops within the same district and year.
To avoid training climate forecasting models on massive duplicated datasets, we construct an independent, deduplicated **Region-Year Climate Panel**:
- 1 observation per `(Region, Year)`
- Causal historical climate lags: $T_{t-1}, T_{t-2}, R_{t-1}, R_{t-2}, H_{t-1}, H_{t-2}$
- Next-year climate targets: $\text{Target\_Temperature} (t+1), \text{Target\_Rainfall} (t+1), \text{Target\_Humidity} (t+1)$


In [8]:
# Extract unique region-year climate records
climate_panel = raw_df[["Region", "Year", "Temperature", "Rainfall", "Humidity"]].drop_duplicates().sort_values(["Region", "Year"]).copy()

# Exact next-year climate targets (t + 1)
climate_target_lookup = climate_panel[["Region", "Year", "Temperature", "Rainfall", "Humidity"]].rename(columns={
    "Year": "Target_Year",
    "Temperature": "Target_Temperature",
    "Rainfall": "Target_Rainfall",
    "Humidity": "Target_Humidity"
})

climate_panel["Target_Year"] = climate_panel["Year"] + 1
climate_panel = climate_panel.merge(climate_target_lookup, on=["Region", "Target_Year"], how="left")

# Causal climate lags (t-1, t-2)
for lag in (1, 2, 3):
    prev_clim = climate_panel[["Region", "Year", "Temperature", "Rainfall", "Humidity"]].rename(columns={
        "Year": f"Prev_Year_{lag}",
        "Temperature": f"Temp_Lag{lag}",
        "Rainfall": f"Rain_Lag{lag}",
        "Humidity": f"Hum_Lag{lag}"
    })
    climate_panel[f"Prev_Year_{lag}"] = climate_panel["Year"] - lag
    climate_panel = climate_panel.merge(prev_clim, on=["Region", f"Prev_Year_{lag}"], how="left").drop(columns=[f"Prev_Year_{lag}"])

print(f"Deduplicated Region-Year Climate Panel: {climate_panel.shape[0]:,} rows x {climate_panel.shape[1]} columns")
display(climate_panel.head(3))


Deduplicated Region-Year Climate Panel: 15,452 rows x 18 columns
                     Region  Year  ...  Rain_Lag3  Hum_Lag3
0  24 Parganas, West Bengal  1966  ...        NaN       NaN
1  24 Parganas, West Bengal  1967  ...        NaN       NaN
2  24 Parganas, West Bengal  1968  ...        NaN       NaN

[3 rows x 18 columns]


## 5. Next-Year Naive Benchmark Ladder Evaluation (2012–2014 Validation)

We evaluate simple heuristic benchmarks on the **2012–2014 Validation Cohort** predicting `Target_Yield`:
1. **Complete History Cohort ($N=9,966$):** Rows where all 3 prior harvest yields are available.
2. **Full Validation Cohort ($N=10,610$):** All validation rows, using district-crop training median as fallback.

This establishes the baseline benchmark ladder for subsequent machine learning models.


In [10]:
def eval_heuristic(y_true, y_pred, label, n):
    err = y_true - y_pred
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    r2 = 1.0 - np.sum(err**2) / np.sum((y_true - np.mean(y_true))**2)
    valid_map = y_true >= 10.0
    map_err = 100.0 * np.mean(np.abs(err[valid_map] / y_true[valid_map]))
    wape = 100.0 * np.sum(np.abs(err)) / np.sum(y_true)
    return {
        "Benchmark Strategy": label,
        "Cohort": "Complete History" if n == 9966 else "Full Validation",
        "Rows": n,
        "MAE (kg/ha)": round(mae, 2),
        "RMSE (kg/ha)": round(rmse, 2),
        "R2": round(r2, 4),
        "MAP (%)": round(map_err, 2),
        "WAPE (%)": round(wape, 2)
    }

val_comp = val_enriched[val_enriched["History_3yr_Complete"] == 1]
y_comp = val_comp["Target_Yield"].values

benchmarks = []
benchmarks.append(eval_heuristic(y_comp, val_comp["Yield_Lag1"].values, "Naive Persistence: Previous Year (Lag 1)", len(val_comp)))
benchmarks.append(eval_heuristic(y_comp, val_comp["Yield_Rolling3"].values, "Rolling 3-Year Historical Mean", len(val_comp)))
benchmarks.append(eval_heuristic(y_comp, val_comp["Yield_Trend_Naive"].values, "Trend-Adjusted Naive", len(val_comp)))

# Full cohort with fallback
val_fallback = val_enriched["Yield_Rolling3"].fillna(val_enriched["Hist_Region_Crop_Median"]).values
benchmarks.append(eval_heuristic(val_enriched["Target_Yield"].values, val_fallback, "Rolling 3-Year Mean (with Regional Fallback)", len(val_enriched)))

benchmark_df = pd.DataFrame(benchmarks)
print("=== NEXT-YEAR NAIVE BENCHMARK RESULTS (2012-2014 VALIDATION) ===")
display(benchmark_df)


=== NEXT-YEAR NAIVE BENCHMARK RESULTS (2012-2014 VALIDATION) ===
                             Benchmark Strategy  ...  WAPE (%)
0      Naive Persistence: Previous Year (Lag 1)  ...     19.84
1                Rolling 3-Year Historical Mean  ...     19.02
2                          Trend-Adjusted Naive  ...     25.98
3  Rolling 3-Year Mean (with Regional Fallback)  ...     20.19

[4 rows x 8 columns]


## 6. Formal Feature Group Schema and Export

We organize candidate features into explicit groups to support systematic feature ablation and pre-model filtering in Phase 04:
- **`G1_Geography`**: `Region, Crop, Soil Type`
- **`G2_Yield_History`**: `Yield_Lag1, Yield_Lag2, Yield_Lag3, Yield_Lag5, Yield_Rolling3, Yield_Rolling3_Std, Yield_Rolling5, Yield_Change_5Y, Yield_Trend_Velocity, Yield_Relative_Index`
- **`G3_Origin_Climate`**: `Temperature, Rainfall, Humidity`
- **`G4_Climate_Anomalies`**: `Temperature_Anomaly, Rainfall_Anomaly, Humidity_Anomaly`
- **`G5_Agronomic_Chemistry`**: `pH, Nitrogen, Phosphorus, Potassium, Fertilizer, Pesticide, NPK_Total, N_P_Ratio`
- **`G6_Meta_Flags`**: `History_Lag1_Available, History_3yr_Complete, History_5yr_Complete, Is_Post_1990`


In [12]:
feature_groups = {
    "G1_Geography": ["Region", "Crop", "Soil Type"],
    "G2_Yield_History": [
        "Yield_Lag1", "Yield_Lag2", "Yield_Lag3", "Yield_Lag5",
        "Yield_Rolling3", "Yield_Rolling3_Std", "Yield_Rolling5",
        "Yield_Change_5Y", "Yield_Trend_Velocity", "Yield_Relative_Index"
    ],
    "G3_Origin_Climate": ["Temperature", "Rainfall", "Humidity"],
    "G4_Climate_Anomalies": ["Temperature_Anomaly", "Rainfall_Anomaly", "Humidity_Anomaly"],
    "G5_Agronomic_Chemistry": ["pH", "Nitrogen", "Phosphorus", "Potassium", "Fertilizer", "Pesticide", "NPK_Total", "N_P_Ratio"],
    "G6_Meta_Flags": ["History_Lag1_Available", "History_3yr_Complete", "History_5yr_Complete", "Is_Post_1990"]
}

# Export enriched panels and climate panel
train_enriched.to_csv(PROCESSED_DIR / "train_forecast_features.csv", index=False)
val_enriched.to_csv(PROCESSED_DIR / "val_forecast_features.csv", index=False)
test_enriched.to_csv(PROCESSED_DIR / "test_forecast_features.csv", index=False)
climate_panel.to_csv(PROCESSED_DIR / "region_year_climate_panel.csv", index=False)

with open(PROCESSED_DIR / "feature_groups_schema.json", "w") as f:
    json.dump(feature_groups, f, indent=2)

print("Saved feature-engineered forecast panels, climate panel, and schema successfully to data/processed/")


Saved feature-engineered forecast panels, climate panel, and schema successfully to data/processed/


## 7. Research Summary and Handoff

1. **Hypothesis-Driven Feature Engineering:** Engineered 10 controlled feature groups spanning exact yield lags, rolling harvest metrics, district climate anomalies, regional baselines, and agronomic ratios.
2. **Dedicated Climate Panel:** Generated `region_year_climate_panel.csv` (1 row per district-year) with next-year climate targets ($T_{t+1}, R_{t+1}, H_{t+1}$) for Phase 06.
3. **Naive Benchmark Established:** The 3-year rolling harvest mean achieves **314.66 kg/ha MAE** ($R^2 = 0.8944$) on the complete-history validation cohort, establishing the competitive benchmark to beat.
4. **Handoff to Phase 04:** The engineered panels and schema provide the inputs for descriptive pre-model feature selection.
